## **Train Custom Yolov5 Object Detection Model**

##### Estimated time: ~ 45-50 minutes
Accelerator: T4 GPU


**Learning objectives:**

1. Use Ultralytics YOLOv5n object detection model for Transfer learning.  Refer to [YOLOv5 quickstart guide](https://github.com/ultralytics/ultralytics/blob/main/docs/en/yolov5/quickstart_tutorial.md) for more details
2. Train object detection model with 128 labeled images
3. Perform inference using trained mode;
4. Save trained model in ONNX format
5. Label own dataset usign labelImg or equivalent tool
6. Train custom object detection model (step 2 above)
7. Save model in ONNX format (similar to step 4 above)

##### **Step 0:**

- Install Ultralytics (this may take some time!)
- Upload labeled images for training

In [0]:
!pip install -q ultralytics
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/object-detect.zip
!unzip -q object-detect.zip

##### **Steps 1,2:**

- Train new object detection model using Yolov5n as base model
- All images resized to 640x640 pixels
- Training epochs = 5 *(Experiment with number of epochs!)*
- View the ***runs*** directory to see training metrics and saved best.pt model

Start off by creating the dataset.yaml file, by populating details of training path and number of classes.

In [0]:
import yaml
import os

# Ensure the directory exists first to avoid FileNotFoundError
data_dir = 'object-detect'
os.makedirs(data_dir, exist_ok=True)

data_yaml = {
    'path': data_dir, 
    'train': 'coco128/images/train2017',
    'val': 'coco128/images/train2017',
    'nc': 80,
    # Wrapped in strings and simplified to a list
    'names': [
        'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 
        'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 
        'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 
        'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 
        'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 
        'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 
        'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 
        'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 
        'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 
        'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
}

yaml_path = os.path.join(data_dir, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("YAML file updated for right file paths!")

We use Transfer Learning to train the model.  Carefully study the results after training. 

In [0]:
from ultralytics import YOLO

# Load a model
model = YOLO('yolov5n.pt')  # load a pretrained model (leverage Transfer training)

# Train the model
results = model.train(data='object-detect/dataset.yaml', epochs=5, imgsz=640)

##### **Discussion:**

Even though the dataset is small, why do you think mAP50 (IoU Threshold = 0.50: The "50" refers to an Intersection over Union (IoU) threshold of 0.50) . This means that if the overlap between the predicted box and the actual object is 50% or higher, the prediction is considered a "True Positive") is **high?** 

##### Solution

<details>
    <summary> Click here for our answer </summary>

    - The value of mAP50 is high because we have used a pretrained model on the entire COCO dataset
    
</details>

##### **Step 3:**

- Perform inference on trained model using image using best trained model.
- Get more details about Ultralytics [YOLO Inference APIs](https://docs.ultralytics.com/modes/predict/)

In [0]:
from ultralytics import YOLO
import cv2
import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt

# Set a relative working directory
base_dir = Path(".")

# Load a model using a relative path
model_path = base_dir / "runs" / "detect" / "train5" / "weights" / "best.pt"
model = YOLO(str(model_path)) 
# Path to best.pt from runs directory

# Download the image
img_url = 'https://ultralytics.com/images/bus.jpg'
img_path = 'bus.jpg'
urllib.request.urlretrieve(img_url, img_path)

# Read the image using cv2.imread
img_input = cv2.imread(img_path)
cv2_imshow(img_input)

# Run inference on one or a list of images
results = model.predict(img_input)

for result in results:
  annotated_image = result.plot()
  plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()


##### **Step 4:**

  - Export best model to ONNX format
  - Model formats to export trained model. Ultralytics [Export option](https://docs.ultralytics.com/modes/export/)
  - Exported model saved as best.onnx in ***runs/detect/train/weights*** directory.
  - Download best.onnx to your local machine
  - Check best.onnx model graph using Netron

In [0]:
from ultralytics import YOLO

# Load a model
model_path = base_dir / "runs" / "detect" / "train5" / "weights" / "best.pt"
model = YOLO(str(model_path))


# Export the model to ONNX format
model.export(format='onnx')

##### **Step 5:**

- Label your own custom dataset using LabelImg. Watch video to get started
- Save your labels and images in the format as the example above
- Create a dataset.yaml file and populate it with location of images and labels folder. The labels file contains classes and bounding boxes of objects from your custom dataset.  
- Train your custom model using a YOLO base model

##### How to create bounding boxes for new dataset
[Watch the video](https://youtube.com/embed/p0nR2YsCY_U)


